# Notebook 16: Positional Encoding & RoPE

**Series:** LLM Systems & Inference  
**Prerequisites:** Notebook 02 (Transformer Architecture), Notebook 05 (KV Cache)

---

## Background: Why Position Matters (and Is Hard)

Transformers are fundamentally **permutation invariant**: if you shuffle the input tokens, the self-attention mechanism produces the same result for each token, just in a different order. This is unlike RNNs, which process tokens sequentially and inherently know which came first.

For language, position is critical. "Dog bites man" is not the same as "Man bites dog." Without some way to inject positional information, a transformer literally cannot tell them apart.

The original 2017 "Attention Is All You Need" paper used **sinusoidal positional encodings**: fixed sine/cosine patterns added to token embeddings before the first layer. This worked, but had a critical limitation: the model learned to use position up to the training sequence length, and extrapolated poorly beyond it.

### The Long-Context Crisis (2021–2022)

As practitioners pushed GPT-style models toward longer contexts (documents, codebases, long conversations), a frustrating pattern emerged: models trained on 2048-token sequences would fail catastrophically at 2049 tokens. Not gradually degrade — they'd produce nonsense, forget the earlier parts of the context, or output repetitive loops.

The root cause was positional encoding. The model had never seen position 2049 during training, so it had no learned sense of what that position meant. Absolute positional encodings don't generalize.

### RoPE: The Solution the Community Converged On (2021)

In 2021, Jianlin Su at ZhuiyiTechnology published ["RoFormer: Enhanced Transformer with Rotary Position Embedding"](https://arxiv.org/abs/2104.09864). The paper introduced **RoPE (Rotary Positional Embedding)**, and it quietly became the most important architectural improvement in modern LLMs.

RoPE is now used in:
- **LLaMA** (all versions)
- **Mistral** and **Mixtral**
- **Gemma** (Google)
- **Qwen** (Alibaba)
- **Falcon**
- **Phi** (Microsoft)

Essentially every major open-source model from 2022 onward uses RoPE.

### Why RoPE Won

RoPE has three key properties that made it dominant:

1. **Relative position** emerges naturally: the dot product `q_i · k_j` depends only on `i - j` (the relative distance), not on the absolute positions `i` and `j`. This is crucial for generalization.

2. **No extra parameters**: RoPE is applied to Q and K at attention time, not embedded in the token representations. It adds zero learned parameters.

3. **Compatible with KV caching**: Because positions are encoded in Q and K (not in the hidden states), the KV cache can store pre-rotated K vectors. Queries for new tokens only need the rotation for their specific position.

### The Math Intuition

RoPE encodes position by **rotating** the query and key vectors in 2D planes. Each pair of dimensions `(d_{2i}, d_{2i+1})` gets rotated by angle `m * θ_i`, where `m` is the token position and `θ_i = 10000^(-2i/d)` (borrowed from the original sinusoidal encoding).

When you compute `q_m · k_n` (query at position m dotted with key at position n), the rotation math works out so that the result depends only on `m - n`, not on `m` or `n` separately.

---

## What You'll Build

1. **Sinusoidal embeddings** (original Transformer approach) — implement and visualize
2. **Learned absolute embeddings** (GPT-2/BERT style) — understand the generalization limit
3. **RoPE from scratch** — implement rotation matrices and apply to Q/K
4. **Relative position proof** — verify that q·k depends only on i-j
5. **Attention patterns** — how positional encoding shapes what tokens attend to
6. **Extrapolation** — why models fail beyond training length and what helps

In [ ]:
!pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(42)
print(f"Device: {device}")

## Part 1: Sinusoidal Positional Encoding (Original Transformer)

In [ ]:
def sinusoidal_encoding(seq_len: int, d_model: int) -> torch.Tensor:
    """
    Original Transformer positional encoding (Vaswani et al. 2017).
    PE[pos, 2i]   = sin(pos / 10000^(2i/d_model))
    PE[pos, 2i+1] = cos(pos / 10000^(2i/d_model))
    """
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(seq_len).unsqueeze(1).float()
    # log-space computation for numerical stability
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term[:d_model // 2])
    return pe


# Visualize
SEQ_LEN, D_MODEL = 64, 128
pe = sinusoidal_encoding(SEQ_LEN, D_MODEL)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Full heatmap
ax = axes[0]
im = ax.imshow(pe.numpy(), aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_xlabel('Dimension (d)')
ax.set_ylabel('Position (pos)')
ax.set_title('Sinusoidal Positional Encoding')
plt.colorbar(im, ax=ax)

# A few dimensions over positions
ax2 = axes[1]
for dim in [0, 2, 10, 30, 60]:
    ax2.plot(pe[:, dim].numpy(), label=f'd={dim}')
ax2.set_xlabel('Position')
ax2.set_ylabel('Value')
ax2.set_title('Sinusoidal Patterns (different dims)')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# Cosine similarity between positions
ax3 = axes[2]
pe_norm = F.normalize(pe, dim=-1)
sim = pe_norm @ pe_norm.T
im3 = ax3.imshow(sim.numpy(), aspect='auto', cmap='viridis', vmin=0, vmax=1)
ax3.set_xlabel('Position j')
ax3.set_ylabel('Position i')
ax3.set_title('Cosine Similarity Between Positions')
plt.colorbar(im3, ax=ax3)

plt.suptitle('Sinusoidal Positional Encoding (Original Transformer)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('sinusoidal_pe.png', dpi=120, bbox_inches='tight')
plt.show()

# Key observation: nearby positions are more similar
pos_0 = pe[0]
sims = F.cosine_similarity(pos_0.unsqueeze(0), pe, dim=-1)
print("Cosine similarity to position 0:")
for pos in [0, 1, 5, 10, 20, 32, 63]:
    print(f"  pos {pos}: {sims[pos]:.3f}")

## Part 2: Learned Absolute Positional Embeddings (GPT-2 Style)

GPT-2 uses a simple approach: a learned embedding table, one vector per position.
Simple and effective up to training length, but terrible at extrapolation.

In [ ]:
class LearnedPositionalEmbedding(nn.Module):
    """GPT-2 style: one learned embedding vector per absolute position."""
    def __init__(self, max_seq_len: int, d_model: int):
        super().__init__()
        self.embedding = nn.Embedding(max_seq_len, d_model)
        self.max_seq_len = max_seq_len
    
    def forward(self, seq_len: int) -> torch.Tensor:
        positions = torch.arange(seq_len)
        return self.embedding(positions)
    
    def extrapolate(self, seq_len: int) -> torch.Tensor:
        """Try to get embeddings beyond training length — this fails gracefully here
        but in practice the model has no idea what to do with unseen positions."""
        if seq_len > self.max_seq_len:
            print(f"WARNING: seq_len {seq_len} > max_seq_len {self.max_seq_len} (out of distribution!)")
            # Clamp to max (bad hack some early systems used)
            positions = torch.clamp(torch.arange(seq_len), 0, self.max_seq_len - 1)
        else:
            positions = torch.arange(seq_len)
        return self.embedding(positions)


# Demonstrate extrapolation failure
lpe = LearnedPositionalEmbedding(max_seq_len=512, d_model=64)
nn.init.normal_(lpe.embedding.weight, std=0.02)  # GPT-2 style init

print("Learned Positional Embedding Analysis:")
print(f"  Training max position: {lpe.max_seq_len - 1}")
print(f"  Embedding table size: {lpe.embedding.weight.shape}")
print(f"  Parameters: {lpe.embedding.weight.numel():,}")
print()

# At training length: all good
emb_train = lpe.forward(seq_len=512)
print(f"  In-distribution (512 tokens): shape {emb_train.shape} ✅")

# Beyond training length: positions map to nearest training position (degenerate)
emb_extra = lpe.extrapolate(seq_len=1024)
print(f"  Out-of-distribution (1024 tokens): positions 512-1023 map to pos 511 (degenerate) ❌")
print()

print("Why this matters:")
print("  For tokens at position 512+, the model assigns position 511's embedding")
print("  → All long-range tokens look the same positionally")
print("  → Model has no sense of their relative ordering")
print("  → GPT-2 (2048 max) needed a hard sequence length limit; no graceful degradation")

## Part 3: RoPE — Rotary Positional Embedding

The key insight: instead of adding position to token embeddings, **rotate** the query
and key vectors in 2D planes. The rotation angle depends on position.

For a d-dimensional vector, pair up dimensions: (0,1), (2,3), ..., (d-2, d-1).
Each pair gets rotated by angle `m * θ_k` where:
- `m` = token position
- `θ_k = 1 / (10000^(2k/d))` = base frequency for dimension pair k

In [ ]:
def build_rope_cache(seq_len: int, head_dim: int, base: float = 10000.0) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Precompute cos and sin tables for RoPE.
    Returns (cos, sin) each of shape [seq_len, head_dim // 2].
    """
    # Frequency for each dimension pair
    # θ_k = 1 / (base^(2k/d)) for k in range(d/2)
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    # Position indices
    positions = torch.arange(seq_len).float()
    # Outer product: [seq_len, head_dim//2]
    freqs = torch.outer(positions, theta)  # m * θ_k for each (m, k)
    return freqs.cos(), freqs.sin()


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    Apply RoPE rotation to a vector.
    x: [batch, seq_len, n_heads, head_dim]
    cos, sin: [seq_len, head_dim // 2]
    
    Rotation in 2D: 
      [x1, x2] → [x1*cos - x2*sin, x1*sin + x2*cos]
    """
    head_dim = x.shape[-1]
    # Split into pairs: x1 = even dims, x2 = odd dims
    x1 = x[..., :head_dim // 2]  # first half
    x2 = x[..., head_dim // 2:]  # second half
    
    # Broadcast cos/sin: [seq_len, d//2] → [1, seq_len, 1, d//2]
    cos = cos.unsqueeze(0).unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(2)
    
    # Apply rotation
    rotated_x1 = x1 * cos - x2 * sin
    rotated_x2 = x1 * sin + x2 * cos
    
    return torch.cat([rotated_x1, rotated_x2], dim=-1)


def rope_attention_score(q_pos: int, k_pos: int, head_dim: int = 64, base: float = 10000.0) -> float:
    """
    Compute a single attention score q[pos_q] · k[pos_k] with RoPE.
    Show that it depends only on (q_pos - k_pos).
    """
    cos_q, sin_q = build_rope_cache(q_pos + 1, head_dim, base)
    cos_k, sin_k = build_rope_cache(k_pos + 1, head_dim, base)
    
    # Random q and k vectors (same for all positions to isolate position effect)
    torch.manual_seed(0)
    q = torch.randn(1, 1, 1, head_dim)
    k = torch.randn(1, 1, 1, head_dim)
    
    q_rot = apply_rope(q, cos_q[q_pos:q_pos+1], sin_q[q_pos:q_pos+1])
    k_rot = apply_rope(k, cos_k[k_pos:k_pos+1], sin_k[k_pos:k_pos+1])
    
    return (q_rot * k_rot).sum().item()


# Demonstrate RoPE
SEQ_LEN = 64
HEAD_DIM = 64

cos_cache, sin_cache = build_rope_cache(SEQ_LEN, HEAD_DIM)

print(f"RoPE cache shapes: cos={cos_cache.shape}, sin={sin_cache.shape}")
print(f"  Each position gets {HEAD_DIM//2} cos values and {HEAD_DIM//2} sin values")
print(f"  These rotate each of the {HEAD_DIM//2} dimension pairs")

# Apply RoPE to a batch
batch, n_heads = 2, 4
x = torch.randn(batch, SEQ_LEN, n_heads, HEAD_DIM)
x_rotated = apply_rope(x, cos_cache, sin_cache)

print(f"\nInput shape:    {x.shape}")
print(f"Rotated shape:  {x_rotated.shape}  (same shape — rotation is in-place geometrically)")
print(f"Norm preserved: {x[0,0,0].norm():.4f} → {x_rotated[0,0,0].norm():.4f}  (rotation preserves norm ✅)")

## Part 4: Proving Relative Position Dependence

The key property of RoPE: the attention score q_m · k_n depends only on (m - n), not on m and n separately.

In [ ]:
def rope_score_at_distance(distance: int, q_pos: int, head_dim: int = 64) -> float:
    """Compute q[q_pos] · k[q_pos - distance] with RoPE."""
    k_pos = q_pos - distance
    if k_pos < 0:
        return float('nan')
    
    max_pos = max(q_pos, k_pos) + 1
    cos, sin = build_rope_cache(max_pos, head_dim)
    
    torch.manual_seed(999)
    q = torch.randn(1, 1, 1, head_dim)
    k = torch.randn(1, 1, 1, head_dim)
    
    q_rot = apply_rope(q, cos[q_pos:q_pos+1], sin[q_pos:q_pos+1])
    k_rot = apply_rope(k, cos[k_pos:k_pos+1], sin[k_pos:k_pos+1])
    return (q_rot * k_rot).sum().item()


# Verify: same distance, different absolute positions → same score
print("Verifying RoPE relative position property:")
print("q[m] · k[m-d] should equal q[m'] · k[m'-d] for any m, m' (same distance d)")
print()
print(f"{'Distance':>10} {'q@10 · k@10-d':>16} {'q@50 · k@50-d':>16} {'q@100 · k@100-d':>18} {'Match?'}")
print("-" * 70)
for dist in [0, 1, 5, 10, 20, 50]:
    s10  = rope_score_at_distance(dist, q_pos=10)
    s50  = rope_score_at_distance(dist, q_pos=50)
    s100 = rope_score_at_distance(dist, q_pos=100)
    match = abs(s10 - s50) < 1e-4 and abs(s10 - s100) < 1e-4
    print(f"{dist:>10} {s10:>16.4f} {s50:>16.4f} {s100:>18.4f} {'✅' if match else '❌'}")

print()
print("→ Scores depend ONLY on distance, not absolute position.")
print("→ This is the key to generalization: a model trained on 2048 tokens")
print("  knows what 'distance 5' means regardless of absolute position.")

In [ ]:
# Visualize how RoPE attention scores decay with distance
distances = list(range(0, 128))

# Compare RoPE vs. sinusoidal
def sinusoidal_score_at_distance(distance: int, q_pos: int = 50, head_dim: int = 64) -> float:
    """Sinusoidal PE: add to embeddings, then dot product — scores depend on absolute pos."""
    k_pos = q_pos - distance
    if k_pos < 0:
        return float('nan')
    
    # For sinusoidal PE, score depends on absolute positions, not just distance
    pe_q = sinusoidal_encoding(q_pos + 1, head_dim)[q_pos]
    pe_k = sinusoidal_encoding(k_pos + 1, head_dim)[k_pos]
    
    torch.manual_seed(999)
    q = torch.randn(head_dim) + pe_q
    k = torch.randn(head_dim) + pe_k
    return (q * k).sum().item() / head_dim


rope_scores = [rope_score_at_distance(d, q_pos=64) / 64 for d in distances]
sin_scores_50 = [sinusoidal_score_at_distance(d, q_pos=50) for d in distances]
sin_scores_500 = [sinusoidal_score_at_distance(d, q_pos=500) for d in distances]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
valid = [(d, s) for d, s in zip(distances, rope_scores) if not np.isnan(s)]
ax.plot([d for d, _ in valid], [s for _, s in valid], color='#e74c3c', linewidth=2, label='RoPE')
ax.set_xlabel('Distance (m - n)')
ax.set_ylabel('Normalized Attention Score')
ax.set_title('RoPE: Score Decays Smoothly with Distance')
ax.legend()
ax.grid(alpha=0.3)

ax2 = axes[1]
valid50  = [(d, s) for d, s in zip(distances, sin_scores_50)  if not np.isnan(s)]
valid500 = [(d, s) for d, s in zip(distances, sin_scores_500) if not np.isnan(s)]
ax2.plot([d for d,_ in valid50],  [s for _,s in valid50],  label='Sinusoidal (abs pos 50)',  color='#3498db')
ax2.plot([d for d,_ in valid500], [s for _,s in valid500], label='Sinusoidal (abs pos 500)', color='#e67e22')
ax2.set_xlabel('Distance (m - n)')
ax2.set_ylabel('Normalized Attention Score')
ax2.set_title('Sinusoidal PE: Scores Depend on Absolute Position')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('RoPE vs Sinusoidal PE: Attention Score vs Distance', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('rope_vs_sinusoidal.png', dpi=120, bbox_inches='tight')
plt.show()

## Part 5: RoPE in a Full Attention Layer

In [ ]:
class RoPEAttention(nn.Module):
    """
    Multi-head attention with RoPE positional encoding.
    This is the attention mechanism used in LLaMA, Mistral, and most modern LLMs.
    """
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int = 2048, rope_base: float = 10000.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        self.wqkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
        
        # RoPE cache — precomputed at init, can be extended for longer sequences
        cos, sin = build_rope_cache(max_seq_len, self.head_dim, base=rope_base)
        self.register_buffer('cos_cache', cos)
        self.register_buffer('sin_cache', sin)
    
    def forward(
        self,
        x: torch.Tensor,  # [batch, seq_len, d_model]
        start_pos: int = 0,  # for KV-cache: position of first token in this batch
        mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        B, T, C = x.shape
        
        # Project to Q, K, V
        qkv = self.wqkv(x)
        q, k, v = qkv.split(C, dim=-1)
        
        # Reshape: [B, T, C] → [B, T, n_heads, head_dim]
        q = q.view(B, T, self.n_heads, self.head_dim)
        k = k.view(B, T, self.n_heads, self.head_dim)
        v = v.view(B, T, self.n_heads, self.head_dim)
        
        # Apply RoPE to Q and K (NOT to V)
        cos = self.cos_cache[start_pos:start_pos + T]
        sin = self.sin_cache[start_pos:start_pos + T]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        
        # Attention: [B, n_heads, T, head_dim]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        scale = self.head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        if mask is not None:
            attn = attn + mask[:T, :T]
        attn = F.softmax(attn, dim=-1)
        
        # Combine heads
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.wo(out)


# Test the layer
D_MODEL, N_HEADS, SEQ_LEN = 256, 4, 32
rope_attn = RoPEAttention(D_MODEL, N_HEADS)

x = torch.randn(2, SEQ_LEN, D_MODEL)
causal_mask = torch.triu(torch.full((SEQ_LEN, SEQ_LEN), float('-inf')), diagonal=1)
out = rope_attn(x, mask=causal_mask)

print(f"RoPE Attention:")
print(f"  Input:  {x.shape}")
print(f"  Output: {out.shape}")
print(f"  Parameters: {sum(p.numel() for p in rope_attn.parameters()):,}")

# No learned positional parameters!
non_rope_params = sum(p.numel() for p in rope_attn.parameters())
rope_overhead = sum(b.numel() for b in rope_attn.buffers())  # cos/sin cache
print(f"  RoPE cache (buffers, not learned): {rope_overhead:,} floats")
print(f"  → RoPE adds ZERO learned parameters")

## Part 6: Extrapolation and RoPE Scaling

Even with RoPE's relative position property, models still degrade at lengths longer than
their training context. This is because the model has never seen *that many* rotations
stacked together. The `base` parameter determines the maximum "useful" frequency range.

The community developed several approaches to extend context without retraining:
- **Linear interpolation** (Su et al. 2023): Scale down positions to fit in training range
- **YaRN** (Peng et al. 2023): Interpolation with NTK-aware frequency adjustment
- **LongRoPE**: Context-adaptive scaling for very long sequences

Notebook 17 covers these in depth. Here we demonstrate why the base matters.

In [ ]:
def effective_wavelength(head_dim: int, base: float) -> np.ndarray:
    """
    The wavelength (period) of the rotation for each dimension pair.
    wavelength_k = 2π / θ_k = 2π * base^(2k/d)
    
    Dimensions with wavelength >> max_seq_len are effectively unused
    (they barely rotate even across the full context).
    """
    k = np.arange(head_dim // 2)
    theta_k = 1.0 / (base ** (2 * k / head_dim))
    wavelength = 2 * np.pi / theta_k
    return wavelength


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compare wavelengths for different bases
HEAD_DIM = 128
bases = {'θ=10000 (LLaMA-1)': 10000, 'θ=500000 (LLaMA-3)': 500000, 'θ=1000000': 1000000}

ax = axes[0]
for label, base in bases.items():
    wl = effective_wavelength(HEAD_DIM, base)
    ax.semilogy(range(HEAD_DIM // 2), wl, label=label, linewidth=2)

# Mark common context lengths
for ctx_len, name in [(2048, '2K'), (8192, '8K'), (128000, '128K')]:
    ax.axhline(ctx_len, color='gray', linestyle=':', alpha=0.6)
    ax.text(HEAD_DIM // 2 - 1, ctx_len * 1.5, f'{name} context', 
            ha='right', fontsize=8, color='gray')

ax.set_xlabel('Dimension pair index (k)')
ax.set_ylabel('Wavelength (tokens)')
ax.set_title('RoPE Wavelength per Dimension Pair')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

# Effective dimensions at different context lengths
ax2 = axes[1]
context_lengths = np.logspace(np.log10(512), np.log10(1e6), 100)
for base_name, base in bases.items():
    wl = effective_wavelength(HEAD_DIM, base)
    # A dimension pair is "effective" if wavelength < context_length
    effective_dims = [np.sum(wl < ctx) for ctx in context_lengths]
    ax2.semilogx(context_lengths, effective_dims, label=base_name, linewidth=2)

ax2.axhline(HEAD_DIM // 2, color='black', linestyle='--', label=f'Max ({HEAD_DIM//2} pairs)')
ax2.set_xlabel('Context Length')
ax2.set_ylabel('"Effective" Dimension Pairs')
ax2.set_title('Effective Dimensions vs Context Length')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3, which='both')

plt.suptitle('RoPE Base Frequency and Context Capacity', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('rope_base_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

print("Key insight: LLaMA-1 used base=10000 (designed for 2K context).")
print("LLaMA-3 uses base=500,000 (supports 128K context).")
print("Increasing the base 'stretches' the rotations, allowing more context.")

## Summary

Positional encoding determines how transformers understand token order — and RoPE solved
this problem so well that it became universal in modern LLMs.

### Key Takeaways

**Why Position Is Hard**  
Transformers are permutation-invariant by design. Position must be explicitly injected.
Absolute encodings (sinusoidal or learned) don't generalize beyond training length.

**What RoPE Does**  
RoPE rotates query and key vectors in 2D planes before the dot-product attention.
Each dimension pair gets a different rotation frequency.
The rotation for Q at position m and K at position n results in a score that depends
only on `m - n` — the relative distance.

**Why RoPE Won**  
- Relative position → better generalization
- Zero extra learned parameters (rotation is deterministic given position)
- KV-cache compatible (K vectors can be pre-rotated and cached)
- Higher base frequencies → longer effective context range

**The Base Parameter**  
The `rope_base` controls how fast each frequency rotates. Low base (10000) = good for short contexts.
High base (500000) = good for long contexts. LLaMA-3 uses 500K base to support 128K context.

### What's Next (Notebook 17: Context Length Extension)

Armed with a deep understanding of RoPE, notebook 17 covers what happens when you try
to extend a model beyond its training context: YaRN interpolation, NTK-aware scaling,
and what actually breaks at long context.